<a href="https://colab.research.google.com/github/lincoln-bot/Assignment-2-HASTS206-GROUP-6/blob/main/assignment2_demographic_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
#MEMBER 1
from google.colab import files
uploaded = files.upload()

Saving Assignment_2_Dataset.zip to Assignment_2_Dataset.zip


In [10]:
import zipfile
import os
zip_path = 'Assignment_2_Dataset.zip'
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
   zip_ref.extractall('assignment_data')
print('Data_set extracted successfully!')
os.listdir('assignment_data')


Data_set extracted successfully!


['insured_transitions.csv',
 'projection_scenarios.csv',
 'life_table_inputs.csv',
 'districts.csv',
 'README.txt',
 'age_sex.csv']

In [11]:
with open('assignment_data/README.txt', 'r') as file:
    readme = file.read()
print(readme)

HASTS206 | Assignment 2 | Dataset guide

All records are synthetic and describe hypothetical districts and insured lives
in a Zimbabwean teaching case. Dates and district labels are illustrative.
Extract this ZIP and keep the original CSV files for reproducibility.
Missing values are blank. Counts represent people unless stated otherwise.

districts.csv (24 rows, key district_id)
  population_2020 and population_2026 are opening stocks on 1 January.
  The opening 2025 stock is obtained by summing age_sex.csv.
  The four event counts cover calendar 2025 and close the balance at 1 Jan 2026.
  piped_water_pct is a district household percentage. Two values are missing.
  region is a synthetic reporting label.

age_sex.csv (864 rows, key district_id + age_start + sex)
  age_start gives 5-year bands (0 = 0-4, 5 = 5-9, ..., 85 = 85+).
  sex is F or M. population_2025 is the 1 Jan 2025 count.
  baseline_deaths_2025 is for residents present at baseline, classified by age
  at baseline. District

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.linalg import expm
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
print('All Libraries imported successfully!')
districts = pd.read_csv('assignment_data/districts.csv')
age_sex = pd.read_csv('assignment_data/age_sex.csv')
life_table = pd.read_csv('assignment_data/life_table_inputs.csv')
scenarios = pd.read_csv('assignment_data/projection_scenarios.csv')
insured = pd.read_csv('assignment_data/insured_transitions.csv')
print('All five datasets loaded successfully!')

All Libraries imported successfully!
All five datasets loaded successfully!


In [13]:
import os, glob
import pandas as pd

folder = "assignment_data"
files = glob.glob(folder + "/*.csv")
data = {}
log = []

# 1. Load original datasets and inspect them
for file in files:
    name = os.path.basename(file)
    df = pd.read_csv(file)
    data[name] = df

    log.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing Values": int(df.isna().sum().sum()),
        "Duplicate Rows": int(df.duplicated().sum()),
        "Numeric Columns": sum(
            pd.api.types.is_numeric_dtype(df[c]) for c in df.columns
        )
    })

    print(f"\n{name}")
    print("Columns:", list(df.columns))
    print("Data types:\n", df.dtypes)

# 2. Save data-quality log
quality_log = pd.DataFrame(log)
os.makedirs("outputs", exist_ok=True)
quality_log.to_csv("outputs/before_analysis_quality_log.csv", index=False)

# 3. Check possible identifier columns
for name, df in data.items():
    for col in df.columns:
        if col.lower() in ["district_id", "district", "id"]:
            print(name, col, "Unique:", df[col].nunique(),
                  "Duplicates:", df[col].duplicated().sum())

# 4. Check numerical ranges
for name, df in data.items():
    print(f"\nNumerical ranges: {name}")
    print(df.select_dtypes(include="number").describe())

print("\nBEFORE ANALYSIS CHECKS COMPLETED.")
print("Quality log saved in outputs folder.")


insured_transitions.csv
Columns: ['district_id', 'origin', 'destination', 'exposure_py', 'transition_count', 'initial_lives']
Data types:
 district_id         object
origin              object
destination         object
exposure_py          int64
transition_count     int64
initial_lives        int64
dtype: object

projection_scenarios.csv
Columns: ['scenario', 'fertility_multiplier', 'arrivals_multiplier', 'departures_multiplier', 'newborn_survival_f', 'newborn_survival_m']
Data types:
 scenario                  object
fertility_multiplier     float64
arrivals_multiplier      float64
departures_multiplier    float64
newborn_survival_f       float64
newborn_survival_m       float64
dtype: object

life_table_inputs.csv
Columns: ['age_start', 'interval_years', 'person_years', 'deaths', 'ax']
Data types:
 age_start           int64
interval_years    float64
person_years        int64
deaths              int64
ax                float64
dtype: object

districts.csv
Columns: ['district_id', 'd


The preliminary data-quality checks were executed successfully, generating a comprehensive log (before_analysis_quality_log.csv). The output confirms the structural integrity of the synthetic datasets prior to analysis:

· Row Counts and Keys: The unique identifier checks confirmed that district_id and other primary keys contain no duplicates, ensuring safe join cardinality and preventing row duplication during data merges.
· Valid Ranges and Missing Indicators: The numerical range checks (describe()) validated the data. For example, five-year survival probabilities fall strictly between 0 and 1, interval widths (ax) align with standard demographic assumptions (ranging from ~0.31 for infancy to 2.5), and scenario multipliers are appropriately centered around the base value of 1.0. No impossible values (e.g., negative populations or births) were detected. Missing values were systematically identified and logged.
· Scope Difference: Acknowledged is the scope difference between districts.csv and age_sex.csv. District-level deaths represent total registered events within the geographic boundary, whereas baseline deaths only include those successfully matched to specific age-sex cohorts. They are not forced to reconcile perfectly.
· Data Retention: All original CSV files remain untouched in the source folder. Cleaning decisions (such as the median imputation for the two missing piped-water indicators) are documented and executed downstream in the relevant analytical sections to retain all 24 districts.

In [14]:
# MEMBER 2: DATA PREPARATION AND QUALITY CHECKS

import pandas as pd
import numpy as np
import os, glob

os.makedirs("outputs", exist_ok=True)

# 1. LOAD DATASETS
districts, age_sex, life_table, scenarios, insured = [
    pd.read_csv(glob.glob(f"**/{f}.csv", recursive=True)[0])
    for f in ["districts", "age_sex", "life_table_inputs",
              "projection_scenarios", "insured_transitions"]
]

# 2. CHECK DATA QUALITY
data = {
    "districts": districts,
    "age_sex": age_sex,
    "life_table": life_table,
    "scenarios": scenarios,
    "insured": insured
}

quality = pd.DataFrame([
    [name, len(df), len(df.columns),
     df.isna().sum().sum(), df.duplicated().sum()]
    for name, df in data.items()
], columns=["Dataset", "Rows", "Columns", "Missing", "Duplicates"])

display(quality)
quality.to_csv("outputs/common_data_quality_report.csv", index=False)

# 3. CHECK DATA STRUCTURE AND VALUES
print("Districts:", districts["district_id"].nunique())
print("Age groups:", sorted(age_sex["age_start"].unique()))
print("Sex categories:", age_sex["sex"].unique())
print("\nMissing values:\n", quality[["Dataset", "Missing"]])

# 4. COMBINE DISTRICT AND AGE-SEX DATA
combined = age_sex.merge(
    districts, on="district_id", how="left"
)
combined.to_csv("outputs/combined_age_sex_district_data.csv", index=False)

# 5. POPULATION BALANCING
pop = age_sex.groupby("district_id")["population_2025"].sum()

balance = districts.copy()
balance["opening_2025"] = balance["district_id"].map(pop)

balance["reconstructed_2026"] = (
    balance["opening_2025"] + balance["births_2025"]
    - balance["deaths_2025"] + balance["arrivals_2025"]
    - balance["departures_2025"]
)

balance["difference"] = (
    balance["reconstructed_2026"] - balance["population_2026"]
)

balance["status"] = np.where(
    balance["difference"].abs() < 0.01, "BALANCED", "REVIEW"
)

display(balance[[
    "district_name", "opening_2025",
    "reconstructed_2026", "population_2026",
    "difference", "status"
]])

balance.to_csv("outputs/population_balance_check.csv", index=False)

print("\nMEMBER 2 DATA PREPARATION COMPLETED.")

,Dataset,Rows,Columns,Missing,Duplicates
0,districts,24,10,2,0
1,age_sex,864,9,0,0
2,life_table,18,5,2,0
3,scenarios,3,6,0,0
4,insured,96,6,0,0


Districts: 24
Age groups: [np.int64(0), np.int64(5), np.int64(10), np.int64(15), np.int64(20), np.int64(25), np.int64(30), np.int64(35), np.int64(40), np.int64(45), np.int64(50), np.int64(55), np.int64(60), np.int64(65), np.int64(70), np.int64(75), np.int64(80), np.int64(85)]
Sex categories: ['F' 'M']

Missing values:
       Dataset  Missing
0   districts        2
1     age_sex        0
2  life_table        2
3   scenarios        0
4     insured        0


,district_name,opening_2025,reconstructed_2026,population_2026,difference,status
0,District D01,172445,173752,173752,0,BALANCED
1,District D02,112211,114605,114605,0,BALANCED
2,District D03,77907,77096,77096,0,BALANCED
3,District D04,95654,96331,96331,0,BALANCED
4,District D05,152444,156477,156477,0,BALANCED
5,District D06,59525,59007,59007,0,BALANCED
6,District D07,44047,44329,44329,0,BALANCED
7,District D08,55286,56674,56674,0,BALANCED
8,District D09,107511,106458,106458,0,BALANCED
9,District D10,111451,112379,112379,0,BALANCED



MEMBER 2 DATA PREPARATION COMPLETED.
